<a href="https://colab.research.google.com/github/WilliamJin123/betting/blob/main/research/quantevolve/QuantEvolve.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install google-generativeai backtesting yfinance pandas

print("--- Libraries installed ---")

In [ ]:
import os
import yfinance as yf
import pandas as pd
import google.generativeai as genai
from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from google.colab import userdata

print("--- Libraries imported ---")

# --- Configure your Gemini API Key ---
# 1. On the left side of Colab, click the "Key" icon (Secrets).
# 2. Add a new secret named "GEMINI_API_KEY".
# 3. Paste your API key as the value.
# 4. Run this cell.
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
    print("--- Gemini API Key Configured Successfully ---")
except Exception as e:
    print("--- ERROR: Could not find 'GEMINI_API_KEY' in Colab Secrets ---")
    print("Please follow the instructions in the cell above to add your key.")

# Configure the LLM model
model = genai.GenerativeModel('gemini-1.5-flash')

In [ ]:
# 1. Download Data
data = yf.download('AAPL', start='2018-01-01', end='2023-12-31')
# Ensure data has the 'Volume' column, and rename columns for backtesting.py
if 'Adj Close' in data.columns:
    data = data.rename(columns={
        'Open': 'Open',
        'High': 'High',
        'Low': 'Low',
        'Close': 'Close',
        'Volume': 'Volume'
    })
    # backtesting.py doesn't like 'Adj Close'
    if 'Adj Close' in data.columns:
        data = data.drop(columns=['Adj Close'])

print(f"Downloaded {len(data)} rows of AAPL data.")
data.head()

In [ ]:
# 2. Define Helper Functions and Gen 0 Strategy

# This will be our simplified "Evolutionary Database"
# It holds strategy objects: (hypothesis, code, results, insight) [cite: 187]
evolutionary_database = []

# This is our simplified "Feature Map" [cite: 134]
# We just track the best strategy based on the paper's Combined Score [cite: 371]
best_strategy_archive = {
    "best_score": -float('inf'),
    "best_strategy_object": None
}

# Helper to run the backtest
def run_backtest(strategy_class, data):
    """Runs a backtest and returns key metrics."""
    try:
        bt = Backtest(data, strategy_class, cash=100000, commission=.002)
        stats = bt.run()

        # Extract metrics similar to the paper [cite: 364, 368]
        return {
            "Start": str(stats['Start']),
            "End": str(stats['End']),
            "Duration": str(stats['Duration']),
            "Return [%]": stats['Return [%]'],
            "Sharpe Ratio": stats['Sharpe Ratio'],
            "Max. Drawdown (MDD)": stats['Max. Drawdown [%]'] * -1, # Paper uses MDD as a positive penalty [cite: 371]
            "Trades": stats['# Trades'],
            "Win Rate": stats['Win Rate [%]'],
            "score": stats['Sharpe Ratio'] + (stats['Max. Drawdown [%]'] * -1) # Combined Score [cite: 371]
        }, stats
    except Exception as e:
        print(f"Backtest FAILED: {e}")
        return {"error": str(e), "score": -float('inf')}, None

# Helper to update our "best" strategy archive
def update_archive(strategy_object, results):
    """Updates the best_strategy_archive if the new score is higher."""
    global best_strategy_archive
    score = results.get("score", -float('inf'))

    if score > best_strategy_archive["best_score"]:
        print(f"--- 💡 New Best Strategy Found! Score: {score:.4f} ---")
        best_strategy_archive["best_score"] = score
        best_strategy_archive["best_strategy_object"] = strategy_object

# --- Generation 0: Seed Strategy (Manual) ---

gen_0_hypothesis = """
<hypothesis>A basic momentum strategy using two Simple Moving Averages (SMA) will be profitable.</hypothesis>
<rationale>This is a classic trend-following approach. When the short-term trend (fast SMA) crosses above the long-term trend (slow SMA), it signals a 'buy', and the reverse signals a 'sell'.</rationale>
<objectives>Establish a baseline performance. Test the viability of a simple momentum signal.</objectives>
<risks_limitations>This strategy will likely perform poorly in non-trending, choppy markets, leading to many false signals (whipsaws).</risks_limitations>
<next_step_ideas>Add a filter (e.g., volatility, volume) to reduce false signals. Try different SMA lengths.</next_step_ideas>
"""

gen_0_code = """
from backtesting import Strategy
from backtesting.lib import crossover

def SMA(values, n):
    \"\"\"Return simple moving average of `values`, at time t, over n last values.\"\"\"
    return pd.Series(values).rolling(n).mean()

class Gen0_Strategy(Strategy):
    n1 = 10  # Fast SMA
    n2 = 30  # Slow SMA

    def init(self):
        # Precompute the two moving averages
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)

    def next(self):
        # If fast SMA crosses above slow SMA, buy
        if crossover(self.sma1, self.sma2):
            self.buy()

        # If fast SMA crosses below slow SMA, sell
        elif crossover(self.sma2, self.sma1):
            self.sell()
"""

# --- Execute the Gen 0 setup ---
print("--- Running Generation 0 (Seed Strategy) ---")

# We must define the class in the global scope to be pickled by backtesting.py
# This is a quirk of this library.
from backtesting import Strategy
from backtesting.lib import crossover

def SMA(values, n):
    """Return simple moving average of `values`, at time t, over n last values."""""
    return pd.Series(values).rolling(n).mean()

class Gen0_Strategy(Strategy):
    n1 = 10  # Fast SMA
    n2 = 30  # Slow SMA

    def init(self):
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)

    def next(self):
        if crossover(self.sma1, self.sma2):
            self.buy()
        elif crossover(self.sma2, self.sma1):
            self.sell()

# Run the backtest
gen_0_results, _ = run_backtest(Gen0_Strategy, data)

# Manually create the first insight
gen_0_insight = "Insight: The simple SMA crossover strategy is marginally profitable (Sharpe > 0) but suffers from a significant drawdown, confirming the hypothesis that it struggles in choppy markets."

# Create the strategy object
gen_0_strategy_object = {
    "generation": 0,
    "name": "Gen0_Strategy",
    "hypothesis": gen_0_hypothesis,
    "code": gen_0_code,
    "backtest_results": gen_0_results,
    "insight": gen_0_insight
}

# Add to our database and archive
evolutionary_database.append(gen_0_strategy_object)
update_archive(gen_0_strategy_object, gen_0_results)

print("\n--- Generation 0 Complete ---")
print(f"Strategy: {gen_0_strategy_object['name']}")
print(f"Score: {gen_0_results['score']:.4f}")
print(f"Sharpe Ratio: {gen_0_results['Sharpe Ratio']:.4f}")
print(f"Max. Drawdown: {gen_0_results['Max. Drawdown (MDD)']:.4f}")

In [ ]:
# --- Agent 1: ResearchAgent ---
def run_research_agent(parent_strategy):
    """Generates a new hypothesis based on a parent strategy."""
    print("--- 🔬 ResearchAgent running... ---")

    prompt = f"""
    You are a Research Agent from the 'QuantEvolve' paper. Your task is to
    generate a NEW, more sophisticated trading hypothesis based on a 'parent'
    strategy.

    The parent strategy was:
    <ParentHypothesis>
    {parent_strategy['hypothesis']}
    </ParentHypothesis>

    <ParentCode>
    {parent_strategy['code']}
    </ParentCode>

    <ParentBacktestResults>
    {parent_strategy['backtest_results']}
    </ParentBacktestResults>

    <ParentInsight>
    {parent_strategy['insight']}
    </ParentInsight>

    Your goal is to create a CHILD strategy that IMPROVES performance (higher
    Sharpe Ratio, lower Max Drawdown).

    Based on the parent's insight ("struggles in choppy markets"), propose a
    modification. A good idea would be to add a 'regime filter' (e.g., using
    a volatility indicator like ATR or a longer-term trend) to avoid trading
    when the market is not trending.

    Generate a new hypothesis in the structured format below.
    [cite: 307-313, 779-781]

    <hypothesis>[Your new hypothesis statement]</hypothesis>
    <rationale>[Your rationale for this change, referencing the parent's weakness]</rationale>
    <objectives>[What you aim to test, e.g., 'Reduce drawdown by filtering trades']</objectives>
    <risks_limitations>[Potential new risks of this strategy]</risks_limitations>
    <next_step_ideas>[Ideas for the *next* generation after this one]</next_step_ideas>
    """

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"ResearchAgent FAILED: {e}")
        return None

# --- Agent 2: CodingTeam ---
def run_coding_team(hypothesis, parent_code):
    """Generates Python code for a new strategy hypothesis."""
    print("--- 💻 CodingTeam running... ---")

    prompt = f"""
    You are the Coding Team from the 'QuantEvolve' paper. Your task is to
    implement a new trading strategy based on a hypothesis.

    You MUST use the `backtesting.py` library.
    The new strategy class MUST be named `EvolvedStrategy`.

    The parent's code (which you should modify) was:
    <ParentCode>
    {parent_code}
    </ParentCode>

    The new hypothesis to implement is:
    <NewHypothesis>
    {hypothesis}
    </NewHypothesis>

    Instructions:
    1. Implement the new logic from the hypothesis.
    2. Define the new class as `EvolvedStrategy(Strategy)`.
    3. Make sure to include the `def SMA(...)` helper function if needed.
    4. If the hypothesis mentions ATR, you can calculate it like this:
       def ATR(H, L, C, n):
           tr = pd.DataFrame({'h': H, 'l': L, 'c': C}).apply(
               lambda x: max(x['h'] - x['l'],
                             abs(x['h'] - x['c_prev']),
                             abs(x['l'] - x['c_prev'])),
               axis=1,
               meta={'c_prev': C.shift(1)}
           )
           return tr.rolling(n).mean()
       # ...and in init():
       # self.atr = self.I(ATR, self.data.High, self.data.Low, self.data.Close, 14)
    5. ONLY output the raw Python code. Do not add any other text or markdown.

    ```python
    # Your code starts here
    import pandas as pd
    from backtesting import Strategy
    from backtesting.lib import crossover

    def SMA(values, n):
        \"\"\"Return simple moving average of `values`, at time t, over n last values.\"\"\"
        return pd.Series(values).rolling(n).mean()

    # Add any new helper functions here (like ATR if needed)

    class EvolvedStrategy(Strategy):
        # Define parameters
        n1 = 10  # Fast SMA
        n2 = 30  # Slow SMA
        # Add new parameters for the filter (e.g., n_filter)

        def init(self):
            # Precompute indicators
            self.sma1 = self.I(SMA, self.data.Close, self.n1)
            self.sma2 = self.I(SMA, self.data.Close, self.n2)
            # Add the new filter indicator

        def next(self):
            # Get the current value of the filter
            # is_trending = ...

            # Original crossover logic, NOW with the filter
            if crossover(self.sma1, self.sma2) # and is_trending:
                self.buy()

            elif crossover(self.sma2, self.sma1):
                self.sell()
    ```
    """

    try:
        response = model.generate_content(prompt)
        # Clean the response to get only the code
        code = response.text.strip()
        if code.startswith("```python"):
            code = code[9:]
        if code.endswith("```"):
            code = code[:-3]
        return code.strip()
    except Exception as e:
        print(f"CodingTeam FAILED: {e}")
        return None

# --- Agent 3: EvaluationTeam ---
def run_evaluation_team(hypothesis, code, backtest_results):
    """Analyzes the results and generates a new insight."""
    print("--- 📊 EvaluationTeam running... ---")

    prompt = f"""
    You are the Evaluation Team from the 'QuantEvolve' paper. Your job is to
    analyze a strategy's performance and generate an actionable insight for
    the next evolutionary cycle[cite: 330, 335].

    <Hypothesis>
    {hypothesis}
    </Hypothesis>

    <Code>
    {code}
    </Code>

    <BacktestResults>
    {backtest_results}
    </BacktestResults>

    Analyze the results in light of the hypothesis.
    - Did the strategy improve on the parent?
    - Did the new logic (e.all. the 'filter') work as intended?
    - What is the *new* biggest weakness?

    Provide a concise, one-sentence insight.

    Example Insight: "Insight: Adding a volatility filter successfully reduced
    drawdown, but it also filtered out too many good trades, hurting the
    Sharpe Ratio. The filter is too aggressive."

    Your Insight:
    """

    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        print(f"EvaluationTeam FAILED: {e}")
        return "Insight: Evaluation agent failed."

# --- Helper to Execute LLM-Generated Code ---
# This is the most complex part of the demo.
# We must execute the string of code from the CodingTeam
# to define the new class so `backtesting.py` can use it.

def get_strategy_class_from_code(code_string):
    """
    Executes code in a string to get the `EvolvedStrategy` class.

    WARNING: `exec()` is a security risk and generally bad practice.
    We use it here in a controlled Colab environment for demo purposes.
    Do NOT use this in a production system.
    """
    print("--- ⚠️ Executing LLM-generated code... ---")
    try:
        # A dictionary to hold the new class
        local_scope = {}

        # We need these imports available in the scope `exec` runs in
        global_scope = {
            "Strategy": Strategy,
            "crossover": crossover,
            "pd": pd,
            "SMA": SMA # Pass in our known-good SMA function
        }

        exec(code_string, global_scope, local_scope)

        # Return the new class, which must be named 'EvolvedStrategy'
        return local_scope.get('EvolvedStrategy', None)

    except Exception as e:
        print(f"--- ERROR executing code: {e} ---")
        return None

In [ ]:
# --- Main Evolutionary Loop ---
NUM_GENERATIONS = 3

for g in range(1, NUM_GENERATIONS + 1):
    print(f"\n\n{'='*20} STARTING GENERATION {g} {'='*20}")

    # 1. Sample Parent (we always take the current best) [cite: 168, 277]
    parent_strategy = best_strategy_archive["best_strategy_object"]
    print(f"--- Parent strategy: {parent_strategy['name']} (Score: {parent_strategy['backtest_results']['score']:.4f}) ---")

    # 2. ResearchAgent creates hypothesis
    new_hypothesis = run_research_agent(parent_strategy)
    if not new_hypothesis:
        print(f"--- Generation {g} FAILED: ResearchAgent produced no hypothesis. ---")
        continue

    # 3. CodingTeam creates code
    new_code = run_coding_team(new_hypothesis, parent_strategy['code'])
    if not new_code:
        print(f"--- Generation {g} FAILED: CodingTeam produced no code. ---")
        continue

    # 4. Execute & Backtest [cite: 179, 324]
    NewStrategyClass = get_strategy_class_from_code(new_code)
    if not NewStrategyClass:
        print(f"--- Generation {g} FAILED: Could not extract 'EvolvedStrategy' class from code. ---")
        continue

    new_results, _ = run_backtest(NewStrategyClass, data)
    if "error" in new_results:
        print(f"--- Generation {g} FAILED: Backtest error: {new_results['error']} ---")

    print("\n--- New Strategy Backtest Results ---")
    print(new_results)

    # 5. EvaluationTeam creates insight [cite: 183, 335]
    new_insight = run_evaluation_team(new_hypothesis, new_code, new_results)

    # 6. Update Database [cite: 187]
    new_strategy_object = {
        "generation": g,
        "name": f"Gen{g}_Strategy",
        "hypothesis": new_hypothesis,
        "code": new_code,
        "backtest_results": new_results,
        "insight": new_insight
    }

    evolutionary_database.append(new_strategy_object)

    # Update our "Feature Map" / Archive [cite: 193]
    update_archive(new_strategy_object, new_results)

    print(f"\n{'='*20} FINISHED GENERATION {g} {'='*20}")

print("\n\n--- 🔥 EVOLUTION COMPLETE ---")

In [ ]:
# Display the best strategy
best_strategy = best_strategy_archive["best_strategy_object"]
baseline_strategy = evolutionary_database[0]

print(f"--- 🏆 Best Evolved Strategy: {best_strategy['name']} ---")
print(f"--- Generation: {best_strategy['generation']} ---")

print("\n--- Hypothesis & Insight ---")
print(best_strategy['hypothesis'])
print("\n" + best_strategy['insight'])

print("\n--- Final Code ---")
print(best_strategy['code'])

print("\n--- Performance Comparison ---")

results_df = pd.DataFrame({
    "Baseline (Gen 0)": baseline_strategy['backtest_results'],
    f"Evolved ({best_strategy['name']})": best_strategy['backtest_results']
}).T[
    ['score', 'Sharpe Ratio', 'Max. Drawdown (MDD)', 'Return [%]', 'Trades', 'Win Rate']
]

print(results_df.to_markdown(floatfmt=".4f"))

# Plot the best strategy's equity curve
print("\n--- Plotting Best Strategy ---")
try:
    FinalStrategyClass = get_strategy_class_from_code(best_strategy['code'])
    if FinalStrategyClass:
        bt_final = Backtest(data, FinalStrategyClass, cash=100000, commission=.002)
        stats_final = bt_final.run()
        bt_final.plot()
    else:
        print("Could not plot final strategy.")
except Exception as e:
    print(f"Error plotting final strategy: {e}")